In [7]:
# Import libraries for computation and analysis
import numpy as np
import numpy.linalg as la
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import sympy as sp

# Display settings for better readability
np.set_printoptions(precision=4, suppress=True)
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)

print("Libraries imported successfully!")
print("Ready to use as a calculator for Assignment 6 computations")

Libraries imported successfully!
Ready to use as a calculator for Assignment 6 computations


In [ ]:
# Problem 3a

centroids = np.array([
    [0,4],
    [6,5]
])
points = np.array([
    [1, 3], [1, 2], [2, 1],
    [2, 2], [2, 3], [3, 2],
    [5, 3], [4, 3], [4, 5],
    [5, 4], [5, 5], [6, 4],
    [6, 5]
])

def d(p,mu):
    return la.norm(p - mu)

distances = np.zeros((len(points), len(centroids)))
assignments = np.zeros(len(points), dtype=int)

print("--- Iteration 1 ---")
for i in range(len(points)):
    for j in range(len(centroids)):
        distances[i, j] = d(points[i], centroids[j])
    assignments[i] = np.argmin(distances[i])
    
    print(f"Point {i+1:2d} {points[i]}: d(m1)={distances[i,0]:.4f}, d(m2)={distances[i,1]:.4f} -> Cluster {assignments[i]+1}")

# Centroid update
new_centroids = np.zeros_like(centroids, dtype=float)
print("\n--- Update Step ---")
for j in range(len(centroids)):
    cluster_points = points[assignments == j]
    if len(cluster_points) > 0:
        new_centroids[j] = np.mean(cluster_points, axis=0)
    else:
        new_centroids[j] = centroids[j]
        
    print(f"New Centroid {j+1}: {new_centroids[j]}")


In [16]:
# Problem 3b

eps = 1.5
min_pts = 2
n = len(points)

dist_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        dist_matrix[i, j] = np.linalg.norm(points[i] - points[j])

neighbors = []
core_points = []

for i in range(n):
    pts_in_eps = np.where(dist_matrix[i] <= eps)[0]
    neighbors.append(pts_in_eps)
    if len(pts_in_eps) >= min_pts:
        core_points.append(i)

clusters = []
visited = set()

for p in core_points:
    if p in visited:
        continue
        
    cluster = []
    queue = [p]
    visited.add(p)
    
    while queue:
        curr = queue.pop(0)
        cluster.append(curr)
        
        if curr in core_points:
            for neighbor in neighbors[curr]:
                if neighbor not in visited:
                    visited.add(neighbor)
                    queue.append(neighbor)
                    
    clusters.append(cluster)

noise = [i for i in range(n) if i not in visited]

for i in range(n):
    print(f"Point {i+1:2d} {points[i]}: neighbors={[n+1 for n in neighbors[i]]}")

print(f"\nCore points: {[p+1 for p in core_points]}")

for idx, c in enumerate(clusters):
    sorted_cluster = sorted([p+1 for p in c])
    print(f"Cluster {idx+1}: {sorted_cluster}")

print(f"Noise: {[p+1 for p in noise]}")

Point  1 [1 3]: neighbors=[1, 2, 4, 5]
Point  2 [1 2]: neighbors=[1, 2, 3, 4, 5]
Point  3 [2 1]: neighbors=[2, 3, 4, 6]
Point  4 [2 2]: neighbors=[1, 2, 3, 4, 5, 6]
Point  5 [2 3]: neighbors=[1, 2, 4, 5, 6]
Point  6 [3 2]: neighbors=[3, 4, 5, 6, 8]
Point  7 [5 3]: neighbors=[7, 8, 10, 12]
Point  8 [4 3]: neighbors=[6, 7, 8, 10]
Point  9 [4 5]: neighbors=[9, 10, 11]
Point 10 [5 4]: neighbors=[7, 8, 9, 10, 11, 12, 13]
Point 11 [5 5]: neighbors=[9, 10, 11, 12, 13]
Point 12 [6 4]: neighbors=[7, 10, 11, 12, 13]
Point 13 [6 5]: neighbors=[10, 11, 12, 13]

Core points: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]
Cluster 1: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]
Noise: []


In [18]:
# Problem 2e: NMI 계산

# 교차표 (Confusion Matrix)
# 행: Cluster 1, 2, 3
# 열: Class X, O, Delta
matrix = np.array([
    [5, 0, 2],  # Cluster 1
    [1, 4, 1],  # Cluster 2
    [0, 2, 4]   # Cluster 3
])

n = np.sum(matrix)

# 1. H(Omega) - 실제 클래스 분포의 엔트로피
class_totals = np.sum(matrix, axis=0) # 열의 합: [6, 6, 7]
p_classes = class_totals / n
h_omega = -np.sum([p * np.log(p) for p in p_classes if p > 0])

# 2. H(C) - 클러스터 할당 분포의 엔트로피
cluster_totals = np.sum(matrix, axis=1) # 행의 합: [7, 6, 6]
p_clusters = cluster_totals / n
h_C = -np.sum([p * np.log(p) for p in p_clusters if p > 0])

# 3. H(Omega | C) - 조건부 엔트로피
h_omega_given_C = 0
for k in range(len(cluster_totals)):
    # 각 클러스터 방 안에서의 속성 비율
    p_class_given_cluster = matrix[k] / cluster_totals[k]
    # 방 안의 엔트로피 계산 (확률이 0인 경우는 0 * ln(0) = 0 으로 취급하여 패스)
    h_cond_k = -np.sum([p * np.log(p) for p in p_class_given_cluster if p > 0])
    
    print(f"Cluster {k+1} internal entropy: {h_cond_k:.4f}")
    # 전체 엔트로피에 클러스터 크기의 가중치를 곱해서 누적
    h_omega_given_C += (cluster_totals[k] / n) * h_cond_k

# 4. Mutual Information & NMI
i_omega_C = h_omega - h_omega_given_C
nmi = i_omega_C / (0.5 * (h_omega + h_C))

print("\n--- Final Results ---")
print(f"H(Omega) = {h_omega:.4f}")
print(f"H(C) = {h_C:.4f}")
print(f"H(Omega | C)  = {h_omega_given_C:.4f}")
print(f"I(Omega ; C)  = {i_omega_C:.4f}")
print(f"NMI = {nmi:.4f}")

Cluster 1 internal entropy: 0.5983
Cluster 2 internal entropy: 0.8676
Cluster 3 internal entropy: 0.6365

--- Final Results ---
H(Omega) = 1.0959
H(C) = 1.0959
H(Omega | C)  = 0.6954
I(Omega ; C)  = 0.4005
NMI = 0.3655


In [17]:
# Problem 3c

n = len(points)
clusters = [[i+1] for i in range(n)]

dist_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        if i != j:
            dist_matrix[i, j] = np.linalg.norm(points[i] - points[j])
        else:
            dist_matrix[i, j] = np.inf

step = 1
while len(clusters) > 1:
    min_dist = np.inf
    m_i, m_j = -1, -1
    
    c_len = len(clusters)
    for i in range(c_len):
        for j in range(i + 1, c_len):
            d = min([dist_matrix[p-1, q-1] for p in clusters[i] for q in clusters[j]])
            if d < min_dist:
                min_dist = d
                m_i, m_j = i, j
                
    c1 = clusters[m_i]
    c2 = clusters[m_j]
    
    print(f"Step {step:2d}: Merge {c1} and {c2} (Distance: {min_dist:.4f})")
    
    new_c = c1 + c2
    clusters.pop(m_j)
    clusters.pop(m_i)
    clusters.append(new_c)
    
    step += 1

Step  1: Merge [1] and [2] (Distance: 1.0000)
Step  2: Merge [3] and [4] (Distance: 1.0000)
Step  3: Merge [5] and [1, 2] (Distance: 1.0000)
Step  4: Merge [6] and [3, 4] (Distance: 1.0000)
Step  5: Merge [7] and [8] (Distance: 1.0000)
Step  6: Merge [9] and [11] (Distance: 1.0000)
Step  7: Merge [10] and [12] (Distance: 1.0000)
Step  8: Merge [13] and [9, 11] (Distance: 1.0000)
Step  9: Merge [5, 1, 2] and [6, 3, 4] (Distance: 1.0000)
Step 10: Merge [7, 8] and [10, 12] (Distance: 1.0000)
Step 11: Merge [13, 9, 11] and [7, 8, 10, 12] (Distance: 1.0000)
Step 12: Merge [5, 1, 2, 6, 3, 4] and [13, 9, 11, 7, 8, 10, 12] (Distance: 1.4142)
